# 01 — First inference: CPU, then GPU

**Question:** how does running the same open-weight model on a GPU change generation time?

We will run `LiquidAI/LFM2.5-1.2B-Instruct`, inspect its inputs and response, and save a small baseline. No training or serving framework is involved. This text-only, 1.17B-parameter checkpoint belongs to the LFM2.5 family [released January 5, 2026](https://www.liquid.ai/blog/introducing-lfm2-5-the-next-generation-of-on-device-ai); its weights use the [LFM Open License](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct).

Run cells in order. Start in Colab with a CPU runtime, then download your result before switching to a GPU runtime and rerunning from the top. Locally, use a Jupyter Python kernel. The first run downloads several GB of weights; FP32 model parameters occupy roughly 4.7 GB (4.4 GiB) of RAM, with additional memory needed for loading and execution.

**Predict first:** will the first request take longer than later requests? Why might generating 64 tokens take longer than processing a 64-token prompt?

Your earlier Qwen CPU notebook is preserved in `archive/01-qwen2.5-cpu.ipynb`, including its displayed outputs. This model has a different tokenizer and architecture, so establish a new CPU baseline before running it on GPU. The earlier Qwen result is not the CPU baseline for an LFM2.5 GPU speedup.


## 1. Install dependencies

**If this session already imported an older Transformers version, restart the session/kernel after installation, then continue from section 2.** Repeating imports does not unload old modules. The setup cell checks for a loaded/installed version mismatch before proceeding.

Keep the runtime's compatible PyTorch installation when it satisfies the requirement. Transformers is pinned for this lesson; exact installed versions are saved with every run. If packages were already imported before installation, restart the kernel/runtime and continue from the next cell.

Use a fresh runtime for this model switch. This lesson now pins Transformers 5.3.0, which includes native LFM2 support; after installing, restart the kernel/runtime if an older version was already imported. The CPU and GPU runs must use the same installed versions.


In [ ]:
# Install the inference libraries in this notebook's Python environment.
%pip install -q "torch>=2.6,<3" "transformers==5.3.0" psutil

## 2. Select hardware and set the experiment

`FORCE_CPU = False` automatically selects CUDA, then Apple MPS, then CPU. Set it to `True` to benchmark CPU even when a GPU is present. Availability means usable by this PyTorch installation; a physically installed GPU may still need a compatible runtime.

Keep the prompt, model revision, precision, token budget, and repeat count unchanged between the CPU and GPU runs. FP32 is deliberate on both devices; lower precision is a later experiment. Changing these settings requires rerunning the following cells, preferably in a fresh runtime.

In [19]:
import csv  # Export each response and its measurements as a table.
import json  # Save complete benchmark results in a structured file.
import os  # Read the number of logical CPU cores.
import platform  # Identify the operating system, CPU, and Python version.
import sys  # Inspect modules already loaded in this notebook kernel.
import statistics  # Summarize repeated measurements with the median.
import time  # Measure elapsed wall-clock time.
import uuid  # Keep filenames unique even when runs finish in the same second.
from datetime import datetime, timezone  # Label result files with a UTC timestamp.
from importlib.metadata import version  # Record the installed library versions.
from pathlib import Path  # Build and create output paths.


# Check for a package upgrade on disk while an older version remains loaded in memory.
loaded_transformers = sys.modules.get("transformers")  # Inspect the cached module without reimporting it.
installed_transformers_version = version("transformers")  # Read the installed package metadata.
if loaded_transformers is not None and loaded_transformers.__version__ != installed_transformers_version:
    raise RuntimeError(  # Stop early with recovery steps rather than mixing package versions.
        f"Transformers {loaded_transformers.__version__} is loaded, but {installed_transformers_version} is installed. "
        "Restart the Colab session/kernel, then rerun from section 2 (Select hardware and set the experiment)."
    )

import transformers  # Load the package after checking for stale imports.
import psutil  # Read the system's physical RAM capacity.
import torch  # Provide tensors, device selection, and inference execution.
from huggingface_hub import model_info  # Resolve a model revision to an exact commit.
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig  # Load the model/tokenizer and configure generation.
from transformers.generation.streamers import BaseStreamer  # Receive generated token IDs before text decoding.

MODEL_NAME = "LiquidAI/LFM2.5-1.2B-Instruct"  # Select the pretrained instruction model.
MODEL_REVISION = "main"  # For the GPU run, paste the resolved revision from the CPU result.
FORCE_CPU = False  # Set True for the CPU baseline even when a GPU is available.
MAX_NEW_TOKENS = 64  # Cap the response length; generation may stop earlier.
MEASURED_RUNS = 3  # Repeat the same request after one warmup.
MEASURE_TOKEN_TIMING = True  # Run an additional instrumented pass for TTFT, TPOT, and token gaps.
PROMPT = "Explain the difference between a CPU and a GPU in three short sentences."  # Keep the workload fixed.
DTYPE = torch.float32  # Use FP32 on both devices for the first comparison.

torch.manual_seed(42)  # Fix the random seed; hardware can still affect numerical results.
torch.set_float32_matmul_precision("highest")  # Avoid reduced-precision FP32 matrix multiplication.
cuda_available = torch.cuda.is_available()  # Check whether PyTorch can use a CUDA GPU.
mps_available = torch.backends.mps.is_available()  # Check whether PyTorch can use an Apple GPU.
device = torch.device("cpu" if FORCE_CPU else "cuda:0" if cuda_available else "mps" if mps_available else "cpu")  # Prefer a GPU unless CPU is forced.

def synchronize():
    # GPU work is asynchronous; wait for it before reading the wall-clock timer.
    if device.type == "cuda":
        torch.cuda.synchronize(device)  # Wait for queued CUDA work to finish.
    elif device.type == "mps":
        torch.mps.synchronize()  # Wait for queued Apple GPU work to finish.

# Save the hardware and software context needed to interpret this benchmark.
environment = {
    "python": platform.python_version(),  # Python interpreter version.
    "packages": {name: version(name) for name in ["torch", "transformers", "huggingface-hub", "tokenizers", "safetensors", "psutil"]},  # Installed dependency versions.
    "loaded_transformers_version": transformers.__version__,  # Record the imported version alongside installed metadata.
    "platform": platform.platform(),  # Operating system and architecture.
    "cpu": platform.processor() or platform.machine(),  # Available CPU identification.
    "logical_cpu_count": os.cpu_count(),  # Logical cores exposed to this process.
    "torch_cpu_threads": torch.get_num_threads(),  # Threads PyTorch uses for CPU operations.
    "system_ram_gib": psutil.virtual_memory().total / 2**30,  # Physical system RAM in GiB.
    "cuda_available": cuda_available,  # Whether CUDA was detected, even if CPU is forced.
    "mps_available": mps_available,  # Whether Apple GPU support was detected.
    "device": str(device),  # Device actually selected for inference.
    "force_cpu": FORCE_CPU,  # Record whether automatic selection was overridden.
    "dtype": str(DTYPE),  # Weight precision used for this comparison.
    "cuda_build": torch.version.cuda,  # CUDA version associated with the PyTorch build.
    "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else "Apple MPS" if device.type == "mps" else None,  # Selected GPU, if any.
    "gpu_total_gib": torch.cuda.get_device_properties(device).total_memory / 2**30 if device.type == "cuda" else None,  # CUDA device capacity; unavailable elsewhere.
}
print(json.dumps(environment, indent=2))  # Inspect the selected environment before loading weights.

{
  "python": "3.13.16",
  "packages": {
    "torch": "2.14.1+cu130",
    "transformers": "5.3.0",
    "huggingface-hub": "1.33.0",
    "tokenizers": "0.22.2",
    "safetensors": "0.8.0",
    "psutil": "5.9.5"
  },
  "loaded_transformers_version": "5.3.0",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "cpu": "x86_64",
  "logical_cpu_count": 8,
  "torch_cpu_threads": 4,
  "system_ram_gib": 50.99027633666992,
  "cuda_available": true,
  "mps_available": false,
  "device": "cuda:0",
  "force_cpu": false,
  "dtype": "torch.float32",
  "cuda_build": "13.0",
  "gpu_name": "Tesla T4",
  "gpu_total_gib": 14.56317138671875
}


## 3. Load the tokenizer and model

The tokenizer maps text to token IDs. The model loads pretrained weights and predicts tokens. We resolve one model revision and use it for both downloads.

The timer below includes revision lookup, any downloads, loading, and moving weights to the selected device. Cached downloads make this time vary; it is kept separate from generation latency. `eval()` disables training behavior such as dropout. No automatic CPU offloading is used: the whole model must fit on the selected device.

LFM2.5 mixes attention layers with gated convolution layers. Attention uses a KV cache; convolution layers retain recent state. The native PyTorch implementation supports execution without optional CUDA convolution kernels. We keep that baseline implementation for both CPU and GPU.


In [20]:
load_start = time.perf_counter()  # Start timing revision lookup, downloads, and model loading.
resolved_revision = model_info(MODEL_NAME, revision=MODEL_REVISION).sha  # Resolve one exact model commit.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=resolved_revision)  # Load the matching text-to-token rules.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,  # Load the chosen model's architecture and pretrained weights.
    revision=resolved_revision,  # Use the same commit as the tokenizer.
    dtype=DTYPE,  # Load weights in the explicitly selected precision.
    attn_implementation="sdpa",  # Use PyTorch's scaled dot-product attention implementation.
).to(device)  # Move the entire model to the selected CPU or GPU.
model.eval()  # Disable training-specific behavior such as dropout.
synchronize()  # Include completed device transfers in the loading measurement.
download_and_load_seconds = time.perf_counter() - load_start  # Keep loading separate from generation timing.

print("Resolved revision:", resolved_revision)  # Copy this revision into the comparison run.
print("Parameter device:", next(model.parameters()).device)  # Confirm where model weights live.
print("Parameter dtype:", next(model.parameters()).dtype)  # Confirm the actual weight precision.
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")  # Count the model's parameters.
print(f"Download + load + device transfer: {download_and_load_seconds:.2f} seconds")  # Display setup time.

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Resolved revision: b9d816ab510f9a28dca37cb182e9da8b0226e13c
Parameter device: cuda:0
Parameter dtype: torch.float32
Parameters: 1,170,340,608
Download + load + device transfer: 3.03 seconds


## 4. Inspect the model inputs

An instruction model expects a chat template that marks message roles and where the assistant response begins. The template itself adds input tokens.

`input_ids` has shape `[batch_size, input_tokens]`. Here the batch contains one request. The attention mask marks real input tokens with `1`; padding tokens would have `0`. Formatting, tokenization, and input transfer happen outside the generation timer.

In [21]:
messages = [{"role": "user", "content": PROMPT}]  # Represent the prompt as a user message.
rendered_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)  # Add message boundaries and the assistant prefix.
print("Formatted prompt:\n", rendered_prompt)  # Inspect the exact text passed to the tokenizer.

# The chat template already includes special tokens; do not add them a second time.
inputs = tokenizer(rendered_prompt, add_special_tokens=False, return_tensors="pt").to(device)  # Tokenize and move tensors to the model's device.
synchronize()  # Finish input transfers before any generation measurement.
input_tokens = inputs["input_ids"].shape[1]  # Read sequence length from [batch_size, sequence_length].
print("Input shape:", tuple(inputs["input_ids"].shape))  # Confirm there is one request in the batch.
print("Input device:", inputs["input_ids"].device)  # Confirm inputs and model use the same device.
print("Input tokens:", input_tokens)  # Include template tokens in the workload count.
print("First 20 token IDs:", inputs["input_ids"][0, :20].tolist())  # Inspect the numeric representation of text.
print("Attention mask:", inputs["attention_mask"][0].tolist())  # Inspect which input positions are real tokens.

Formatted prompt:
 <|startoftext|><|im_start|>user
Explain the difference between a CPU and a GPU in three short sentences.<|im_end|>
<|im_start|>assistant

Input shape: (1, 24)
Input device: cuda:0
Input tokens: 24
First 20 token IDs: [1, 6, 6423, 708, 27388, 958, 779, 5521, 1685, 768, 30110, 810, 768, 54841, 797, 2204, 3290, 19528, 523, 7]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 5. Generate one response

`generate()` repeatedly predicts the next token until an end-of-sequence token or the output limit is reached. Greedy decoding chooses the highest-scoring token. `use_cache=True` reuses attention keys/values and convolution state within this request; it does not cache an answer across requests.

`torch.inference_mode()` disables gradient bookkeeping. The returned sequence includes the input, so we slice it off before counting and decoding the response.

Our timer covers **in-process generation**, including prefill, decoding, and generation-loop overhead. It excludes loading, tokenization, input transfer, decoding IDs to text, and printing. GPU synchronization surrounds the measurement. This is neither API latency nor decode-only speed.

In [22]:
generation_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS,  # Limit newly generated tokens without counting the prompt.
    do_sample=False,  # Choose the highest-scoring next token instead of sampling.
    num_beams=1,  # Generate a single greedy continuation without beam search.
    use_cache=True,  # Reuse attention keys/values and convolution state within this request.
    eos_token_id=model.generation_config.eos_token_id,  # Stop at the model's end-of-sequence token(s).
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,  # Supply a valid padding token ID.
)

class TokenTimingStreamer(BaseStreamer):
    # Timestamp each generated token ID as it reaches Python, without buffering decoded text.
    def __init__(self):
        self.skip_prompt = True  # Transformers sends the complete prompt before the first generated token.
        self.arrivals = []  # Store one absolute wall-clock timestamp per generated token.

    def put(self, value):
        arrived = time.perf_counter()  # Timestamp callback entry, after Transformers has copied IDs to CPU.
        if self.skip_prompt:
            self.skip_prompt = False  # Exclude the initial prompt callback from TTFT and token counts.
            return
        if value.numel() != 1:  # This lesson measures one greedy request, with one token per callback.
            raise ValueError("Token timing requires batch size 1 and one generated token per callback.")
        self.arrivals.append(arrived)  # Keep the timestamp; decode the response only after timing finishes.

    def end(self):
        pass  # Generation is synchronous here, so no queue or background thread needs closing.


def generate_once(measure_token_timing=False):
    # Return both the generated response and measurements from one independent request.
    streamer = TokenTimingStreamer() if measure_token_timing else None  # Keep the original baseline free of token callbacks.
    synchronize()  # Exclude previously queued device work from this request's timer.
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)  # Reset the peak counter, retaining the loaded model allocation.
    start = time.perf_counter()  # Start timing immediately before generation.
    with torch.inference_mode():  # Disable gradient bookkeeping during inference.
        output = model.generate(**inputs, generation_config=generation_config, streamer=streamer)  # Optionally observe token IDs during generation.
    synchronize()  # Wait for generation to finish before stopping the timer.
    seconds = time.perf_counter() - start  # Measure generation wall time, including synchronization.
    generated_ids = output[0, input_tokens:].cpu().tolist()  # Remove prompt tokens and copy response IDs to a Python list.
    result = {
        "input_tokens": input_tokens,  # Record the prompt's actual token length.
        "output_tokens": len(generated_ids),  # Count actual generated IDs, including any terminal token.
        "generation_seconds": seconds,  # Store unrounded generation latency.
        "output_tokens_per_second": len(generated_ids) / seconds,  # Divide generated tokens by generation time.
        "cuda_peak_allocated_gib": torch.cuda.max_memory_allocated(device) / 2**30 if device.type == "cuda" else None,  # Record CUDA peak allocation; leave unavailable metrics null.
        "generated_token_ids": generated_ids,  # Preserve exact output IDs in the JSON results.
        "response": tokenizer.decode(generated_ids, skip_special_tokens=True),  # Preserve readable output beside its measurements.
    }

    if measure_token_timing:
        arrivals = [arrived - start for arrived in streamer.arrivals]  # Convert absolute timestamps to seconds since generation started.
        if len(arrivals) != len(generated_ids):  # Check that every output ID, including EOS, has exactly one timestamp.
            raise RuntimeError("Streamed token count does not match the generated output length.")
        gaps = [later - earlier for earlier, later in zip(arrivals, arrivals[1:])]  # Measure each gap after the first token.
        ttft = arrivals[0] if arrivals else None  # Time to the first generated token ID reaching Python.
        result.update({
            "ttft_seconds": ttft,  # Includes generation setup, prefill, first-token selection, and delivery to the callback.
            "tpot_seconds": (seconds - ttft) / (len(arrivals) - 1) if len(arrivals) > 1 else None,  # Average post-first-token time through generate() completion.
            "time_to_last_token_seconds": arrivals[-1] if arrivals else None,  # Time until the final generated ID reaches Python.
            "mean_itl_seconds": statistics.mean(gaps) if gaps else None,  # Mean observed token-to-token gap; excludes completion cleanup.
            "min_itl_seconds": min(gaps) if gaps else None,  # Shortest gap observed within this request.
            "max_itl_seconds": max(gaps) if gaps else None,  # Longest gap observed within this request.
            "token_arrival_seconds": arrivals,  # Preserve the full per-token timeline in JSON.
            "inter_token_latency_seconds": gaps,  # Preserve all N-1 inter-token gaps in JSON.
        })
    return result  # Return the response together with either baseline or instrumented measurements.


first_request = generate_once()  # Use the first request as a separately recorded warmup.
print(first_request["response"])  # Read the actual model response.
print(f"First request: {first_request['output_tokens']} output tokens in {first_request['generation_seconds']:.3f}s")  # Inspect warmup latency.

A CPU (Central Processing Unit) handles general-purpose tasks and processes multiple instructions sequentially. A GPU (Graphics Processing Unit) specializes in parallel processing for graphics and complex calculations. While CPUs are versatile, GPUs excel at handling large datasets and parallel workloads efficiently.
First request: 56 output tokens in 1.377s


## 6. Warmed-up baseline

The first request above is our warmup and is excluded from the summary. Each repetition starts a new request with a new attention/convolution cache. Reusing the same prompt controls the workload, not a response cache.

Count actual generated token IDs, including a terminal special token if produced. A token cap is not an exact output length. CUDA peak allocated memory includes the loaded model and request allocations; it is not total GPU usage. CPU/MPS peak memory is left unavailable (`null`), rather than reported as zero.

Three repetitions provide a first observation, not a reliable p95 estimate. Do not expect an exact speedup or identical generated text across different hardware.

In [23]:
measurements = []  # Keep every measured response and its metrics, rather than just averages.
for repeat in range(1, MEASURED_RUNS + 1):  # Run the same workload the configured number of times.
    row = {"repeat": repeat, **generate_once()}  # Label this request and retain its full result.
    measurements.append(row)  # Preserve the response, token IDs, latency, throughput, and memory.
    print(f"Run {repeat}: {row['output_tokens']} tokens | {row['generation_seconds']:.3f}s | {row['output_tokens_per_second']:.2f} tokens/s")  # Show this request's measurements.
    print("Response:", row["response"], "\n")  # Inspect each response next to its measurements.

summary = {
    "measured_requests": len(measurements),  # Report the sample count, excluding the warmup.
    "median_generation_seconds": statistics.median(row["generation_seconds"] for row in measurements),  # Summarize warmed-up request latency.
    "median_output_tokens_per_second": statistics.median(row["output_tokens_per_second"] for row in measurements),  # Summarize per-request output rate.
}
print(json.dumps(summary, indent=2))  # Display the summary without discarding individual results.

Run 1: 56 tokens | 1.371s | 40.83 tokens/s
Response: A CPU (Central Processing Unit) handles general-purpose tasks and processes multiple instructions sequentially. A GPU (Graphics Processing Unit) specializes in parallel processing for graphics and complex calculations. While CPUs are versatile, GPUs excel at handling large datasets and parallel workloads efficiently. 

Run 2: 56 tokens | 1.349s | 41.51 tokens/s
Response: A CPU (Central Processing Unit) handles general-purpose tasks and processes multiple instructions sequentially. A GPU (Graphics Processing Unit) specializes in parallel processing for graphics and complex calculations. While CPUs are versatile, GPUs excel at handling large datasets and parallel workloads efficiently. 

Run 3: 56 tokens | 1.352s | 41.42 tokens/s
Response: A CPU (Central Processing Unit) handles general-purpose tasks and processes multiple instructions sequentially. A GPU (Graphics Processing Unit) specializes in parallel processing for graphics and co

## 6b. Token-level timing (optional)

The original baseline above still measures generation without token callbacks. With `MEASURE_TOKEN_TIMING = True`, this extra pass records one warmup and the same number of measured requests, including every response. It roughly doubles the number of inference calls in this notebook.

For N generated token IDs and generation completion time T, with all times measured from the start of `generate()`:

| Metric | Definition in this notebook |
| --- | --- |
| TTFT | Time until the first generated token ID reaches the Python callback. Includes generation setup, prefill, first-token selection, and the CPU copy. |
| TPOT | `(T - TTFT) / (N - 1)`. Average time after the first token through generation completion; `null` for fewer than two output tokens. |
| ITL | Each gap between consecutive token arrivals. There are N-1 gaps. |
| Mean ITL | `(last token arrival - first token arrival) / (N - 1)`. Unlike TPOT above, it excludes cleanup after the final token arrives. |
| Generation latency | Time through completion of `generate()`, including the instrumentation. |

All saved times use **seconds**. Counts include generated special tokens such as EOS. The initial prompt callback is skipped, even if its input has only one token. A first token ID can be a special token; this is not time to the first visible word. TTFT also is not a pure prefill-kernel measurement.

Transformers copies generated IDs to CPU before calling the streamer. That copy and the Python callbacks add overhead. Keep these instrumented results separate from the original baseline; compare CPU and GPU using the same measurement mode. There is no HTTP server, network, request queue, tokenization, or input transfer in these timing scopes. Prefill/decode kernel profiling and API timing are later experiments.

Sources: [Transformers token delivery](https://github.com/huggingface/transformers/blob/v5.3.0/src/transformers/generation/utils.py), [streamer interface](https://github.com/huggingface/transformers/blob/v5.3.0/src/transformers/generation/streamers.py).

In [24]:
token_timing = None  # Leave diagnostics absent if the optional timing pass is disabled.
if MEASURE_TOKEN_TIMING:
    timing_warmup = generate_once(measure_token_timing=True)  # Warm up the instrumented path separately.
    timing_measurements = []  # Preserve every instrumented response and its token timeline.
    for repeat in range(1, MEASURED_RUNS + 1):
        row = {"repeat": repeat, **generate_once(measure_token_timing=True)}  # Measure an independent request with a fresh cache.
        timing_measurements.append(row)  # Retain raw measurements instead of keeping only medians.
        print({key: row[key] for key in ["repeat", "output_tokens", "ttft_seconds", "tpot_seconds", "mean_itl_seconds", "generation_seconds"]})  # Inspect request-level timing in seconds.
    timing_summary = {}  # Aggregate only measured requests, excluding the instrumented warmup.
    for metric in ["ttft_seconds", "tpot_seconds", "mean_itl_seconds", "generation_seconds"]:
        values = [row[metric] for row in timing_measurements if row[metric] is not None]  # Exclude undefined metrics, such as TPOT for one output token.
        timing_summary[f"median_{metric}"] = statistics.median(values) if values else None  # Keep undefined aggregates null.
    token_timing = {
        "version": "token-streamer-v1",  # Identify this measurement method separately from the original baseline.
        "timing_scope": "generate() start to Python receipt of token IDs and generation completion; includes streamer CPU copies and callbacks",  # Record instrumentation overhead in the artifact.
        "tpot_definition": "(generation_seconds - ttft_seconds) / (output_tokens - 1)",  # Include completion cleanup in this request-level average.
        "mean_itl_definition": "(last_token_arrival - first_token_arrival) / (output_tokens - 1)",  # Average only the observed gaps between tokens.
        "first_request": timing_warmup,  # Preserve the instrumented warmup, including its full response.
        "measurements": timing_measurements,  # Preserve responses, metrics, token IDs, and timelines.
        "summary": timing_summary,  # Store warmed-up medians for the instrumented pass.
    }
    print(json.dumps(timing_summary, indent=2))  # Display the additional metrics without replacing the original baseline.

{'repeat': 1, 'output_tokens': 56, 'ttft_seconds': 0.0355864080001993, 'tpot_seconds': 0.0241281894909083, 'mean_itl_seconds': 0.024120618054538766, 'generation_seconds': 1.362636830000156}
{'repeat': 2, 'output_tokens': 56, 'ttft_seconds': 0.03552702900014992, 'tpot_seconds': 0.024232269181819983, 'mean_itl_seconds': 0.024225146327270606, 'generation_seconds': 1.368301834000249}
{'repeat': 3, 'output_tokens': 56, 'ttft_seconds': 0.035945523000009416, 'tpot_seconds': 0.0242443436363663, 'mean_itl_seconds': 0.024235885890911132, 'generation_seconds': 1.369384423000156}
{
  "median_ttft_seconds": 0.0355864080001993,
  "median_tpot_seconds": 0.024232269181819983,
  "median_mean_itl_seconds": 0.024225146327270606,
  "median_generation_seconds": 1.368301834000249
}


## 7. Save this run before switching runtimes

Each benchmark produces two baseline files with the same unique filename stem under `runs/first-inference/LFM2.5-1.2B-Instruct/`, relative to the notebook's working directory:

- **JSON:** environment, prompt/template, model revision, generation settings, baseline responses/metrics/summary, and optional `token_timing` diagnostics containing every instrumented response and per-token timeline.
- **Baseline CSV:** one row per uninstrumented request, including the warmup, with its phase, repeat number, prompt, full response text, token counts, latency, output rate, and CUDA peak memory. Open it in a table viewer to inspect responses beside their measurements. Multiline responses are quoted as a single CSV field.

When diagnostics are enabled, a third file ending in `-token-timing.csv` contains the instrumented requests and TTFT/TPOT/ITL scalar metrics. Full arrival/gap arrays remain in JSON. All artifacts are printed by the preview cell below.

The warmup has phase `warmup` and repeat `0`; benchmark repetitions have phase `measured` and repeats `1` onward. Only measured requests contribute to the summary. No weights are saved here. This directory is ignored by Git.

Run the save cell after benchmarking. In Colab, download all generated files using the optional cell below **before switching runtime**. Runtime storage is temporary. Keep the CPU files for the GPU comparison. Selected reviewed results can later be copied into the repository's `results/` directory.

In [25]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]  # Give this benchmark a unique timestamped identity.
result = {
    "benchmark_version": "first-inference-v1",  # Identify the measurement procedure.
    "run_id": run_id,  # Link JSON and CSV files to the same benchmark.
    "environment": environment,  # Preserve the hardware and software context.
    "model": MODEL_NAME,  # Record which pretrained model ran.
    "resolved_revision": resolved_revision,  # Pin the model/tokenizer version for comparison.
    "attention_implementation": model.config._attn_implementation,  # Record the attention implementation actually configured.
    "messages": messages,  # Preserve the original user prompt and role.
    "rendered_prompt": rendered_prompt,  # Preserve the complete chat-formatted input.
    "input_token_ids": inputs["input_ids"][0].cpu().tolist(),  # Save the exact tokenized workload.
    "generation_config": generation_config.to_dict(),  # Save all generation settings.
    "seed": 42,  # Record the seed used during setup.
    "warmup_requests": 1,  # State how many requests were excluded from the summary.
    "download_and_load_seconds": download_and_load_seconds,  # Keep setup time separate from request latency.
    "timing_scope": "model.generate plus device synchronization; excludes tokenization, input transfer and text decoding",  # Define the generation timer.
    "first_request": first_request,  # Retain the warmup response, token IDs, and measurements.
    "measurements": measurements,  # Retain every measured response, token IDs, and measurements.
    "summary": summary,  # Include the warmed-up medians for quick comparison.
    "token_timing": token_timing,  # Keep optional TTFT/TPOT/ITL measurements separate from the baseline.
}
output_dir = Path("runs") / "first-inference" / MODEL_NAME.split("/")[-1]  # Group ignored benchmark artifacts by model.
output_dir.mkdir(parents=True, exist_ok=True)  # Create the output directory if needed.
result_path = output_dir / f"{device.type}-{run_id}.json"  # Separate CPU/GPU runs with unique filenames.
result_path.write_text(json.dumps(result, indent=2) + "\n")  # Save full results without rounding the measurements.

csv_path = result_path.with_suffix(".csv")  # Pair the readable response table with the JSON file.
fieldnames = ["phase", "repeat", "prompt", "response", "input_tokens", "output_tokens", "generation_seconds", "output_tokens_per_second", "cuda_peak_allocated_gib"]  # Define one row's columns.
requests = [{"phase": "warmup", "repeat": 0, **first_request}] + [{"phase": "measured", **row} for row in measurements]  # Include every response while labeling the warmup.
with csv_path.open("w", newline="", encoding="utf-8") as file:  # Let the CSV writer handle multiline text and quoting.
    writer = csv.DictWriter(file, fieldnames=fieldnames, extrasaction="ignore")  # Keep token-ID arrays in JSON, rather than duplicating them in CSV.
    writer.writeheader()  # Name the columns for table viewers.
    for request in requests:  # Write the warmup and each measured request independently.
        writer.writerow({"prompt": PROMPT, **request})  # Store each full response beside its own measurements.
print("Saved JSON:", result_path.resolve())  # Locate the complete benchmark artifact.
print("Saved CSV:", csv_path.resolve())  # Locate the per-request response table.

timing_csv_path = None  # Only create a diagnostic CSV when the optional pass ran.
if token_timing is not None:
    timing_csv_path = result_path.with_name(result_path.stem + "-token-timing.csv")  # Give instrumented requests their own table.
    timing_fields = fieldnames + ["ttft_seconds", "tpot_seconds", "time_to_last_token_seconds", "mean_itl_seconds", "min_itl_seconds", "max_itl_seconds"]  # Add scalar timing metrics; keep full arrays in JSON.
    timing_requests = [{"phase": "warmup", "repeat": 0, **token_timing["first_request"]}] + [{"phase": "measured", **row} for row in token_timing["measurements"]]  # Label the instrumented warmup separately.
    with timing_csv_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=timing_fields, extrasaction="ignore")  # Serialize full responses with correct CSV quoting.
        writer.writeheader()  # Name the diagnostic columns.
        for request in timing_requests:
            writer.writerow({"prompt": PROMPT, **request})  # Pair each instrumented response with its own metrics.
    print("Saved token-timing CSV:", timing_csv_path.resolve())  # Locate the additional table.

Saved JSON: /content/runs/first-inference/LFM2.5-1.2B-Instruct/cuda-20261122T103323Z-4c863240.json
Saved CSV: /content/runs/first-inference/LFM2.5-1.2B-Instruct/cuda-20261122T103323Z-4c863240.csv
Saved token-timing CSV: /content/runs/first-inference/LFM2.5-1.2B-Instruct/cuda-20261122T103323Z-4c863240-token-timing.csv


**Inspect the saved benchmark files**

Run the next cell after saving to print the complete JSON, baseline CSV, and optional token-timing CSV, including all responses and measurements, with each file's size. It reads the existing files without running inference again. Save the executed notebook to retain this displayed output.


In [26]:
for path in [result_path, csv_path] + ([timing_csv_path] if timing_csv_path is not None else []):  # Inspect baseline files and any token-timing table.
    print(f"\n{path.name} ({path.stat().st_size / 1024:.1f} KiB)")  # Identify the file and report its actual size.
    print(path.read_text(encoding="utf-8"))  # Print the full saved contents, including every response and measurement.



cuda-20261122T103323Z-4c863240.json (30.3 KiB)
{
  "benchmark_version": "first-inference-v1",
  "run_id": "20261122T103323Z-4c863240",
  "environment": {
    "python": "3.13.16",
    "packages": {
      "torch": "2.14.1+cu130",
      "transformers": "5.3.0",
      "huggingface-hub": "1.33.0",
      "tokenizers": "0.22.2",
      "safetensors": "0.8.0",
      "psutil": "5.9.5"
    },
    "loaded_transformers_version": "5.3.0",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "cpu": "x86_64",
    "logical_cpu_count": 8,
    "torch_cpu_threads": 4,
    "system_ram_gib": 50.99027633666992,
    "cuda_available": true,
    "mps_available": false,
    "device": "cuda:0",
    "force_cpu": false,
    "dtype": "torch.float32",
    "cuda_build": "13.0",
    "gpu_name": "Tesla T4",
    "gpu_total_gib": 14.56317138671875
  },
  "model": "LiquidAI/LFM2.5-1.2B-Instruct",
  "resolved_revision": "b9d816ab510f9a28dca37cb182e9da8b0226e13c",
  "attention_implementation": "sdpa",
  "messages": [

In [27]:
# Optional, Colab only: run these lines to download the benchmark artifacts.
from google.colab import files  # Access Colab's browser download helper.
files.download(str(result_path))  # Download the complete JSON results.
files.download(str(csv_path))  # Download the CSV containing every response and its measurements.
if timing_csv_path is not None:  # Download token-level request metrics when available.
    files.download(str(timing_csv_path))  # Download the separate instrumented-request table.

## 8. Repeat on GPU and interpret the result

1. Save the CPU JSON and CSV, and copy its `resolved_revision` into `MODEL_REVISION` above.
2. In Colab, change the runtime hardware accelerator to a GPU. The runtime may restart.
3. Keep `FORCE_CPU = False` and run all cells in order. Confirm the selected device is `cuda:0` and inspect the GPU name. Without an accelerator, the notebook will still use CPU.
4. Download the GPU JSON and CSV. Compare median generation latency and median output tokens/second alongside actual output lengths and responses.
5. Calculate the **observed latency speedup** as `CPU median generation seconds / GPU median generation seconds`. If token counts differ, that ratio also reflects the changed amount of generated work. Compare output tokens/second too and report the difference.

Keep FP32 for this first comparison. CPU and GPU in separate Colab sessions also differ in host resources; this is a comparison of the recorded environments. Reduced precision, fixed exact output lengths, multiple prompts, and broader benchmarking follow in later experiments.

Record your answers after running:

- Selected CPU/GPU hardware and model revision:
- Input tokens and actual generated tokens:
- First-request time versus warmed-up median:
- CPU versus GPU latency and output rate:
- Did output text or token count change?
- What is excluded from the generation timer?
- Why are `eval()` and `inference_mode()` both used?
- What question should the next experiment investigate?

**Sources:** [LFM2.5 model card](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct), [Transformers loading options](https://huggingface.co/docs/transformers/main_classes/model), [PyTorch CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html), [PyTorch MPS backend](https://docs.pytorch.org/docs/stable/notes/mps.html).

Compare LFM2.5 CPU against LFM2.5 GPU. Comparing Qwen CPU against LFM2.5 GPU changes the model, tokenizer, architecture, and hardware together.
